[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Joins and Aggregates


## What you will be able to do

Join classes along their relationships, keep the rows with nothing to join to with an outer join,
and join a table to itself with `aliased`. Count, sum and average with `func` and `group_by`, keep
groups with `having`, and compute grade point averages in SQL without a floating point sum. Filter
on related rows with `any()` and `has()`, build a report from a subquery, and recognize a cartesian
product, counts multiplied by a second collection, a collection filtered like a column, and a
collection that `contains_eager` filled with half its rows.


## The idea

### The problem

The registrar's office asks questions of the whole college at once. How many students is every
section carrying? Which students have never failed a course, and which have no enrollments at all?
What is every student's grade point average, and how many in each program made the dean's list last
term? Loading every object and working in Python answers all of them, slowly and with every row in
memory. The database can answer them itself, with joins, `GROUP BY` and aggregate functions, and send
back one row per answer.

Queries like these fail quietly. A join that leaves out its condition pairs every student with every
course, and returns twenty-five rows where one was meant. A count across two joined collections
counts every combination of the two, and reports a course with four sections as having twenty-three.
An inner join drops the student with no enrollments from a list whose whole purpose was to find that
student.
None of it raises an error, and all of it looks like data.

### What joins and aggregates are

> A **join** combines rows of two tables that match on a condition, which a relationship supplies:
> `.join(Student.enrollments)`. An **outer join**, `.outerjoin()`, keeps the rows of the left side
> that match nothing, with `NULL` in the right side's columns. **`aliased(Enrollment)`** is a second
> name for the same table, for a query that uses it twice. An **aggregate** function, such as
> `func.count()`, `func.sum()` or `func.avg()`, turns many rows into one, once for every group of a
> **`group_by()`**, and **`having()`** filters the groups the way `where()` filters rows.
> **`Student.enrollments.any(...)`** asks whether some related row matches, and
> **`Enrollment.student.has(...)`** whether the one related object does, both as `EXISTS`
> subqueries. **`contains_eager()`** loads a collection from a join the query already makes.

### Why it works that way

- **A relationship is a join condition.** `.join(Student.enrollments)` writes the `ON` clause from
  the foreign key, so the condition cannot be forgotten or mistyped.
- **Two tables with no condition make every pair.** A `FROM` that names two tables without joining
  them is a cartesian product, and SQLAlchemy warns when it compiles one.
- **An inner join keeps only matches.** The student with no enrollments and the section with no
  students vanish from an inner join, and an outer join keeps them, with `NULL` where the other side
  would be.
- **Joins multiply rows.** Joining a course to its sections and its sections to their enrollments
  makes a row for every enrollment, so counting sections in those rows counts enrollments, unless
  the count is `DISTINCT`.
- **`any()` and `has()` test without joining.** An `EXISTS` subquery asks about related rows without
  adding them to the result, so nothing multiplies.
- **Whole numbers add up the same everywhere.** SQLite adds floating point numbers with more care
  since version 3.43, so a sum of them can differ in its last digits from one SQLite to another, and
  a sum of whole numbers cannot. Grade points in tenths keep every sum whole.

### Where this shows up

Every report, dashboard and export runs queries of this shape, and the **Joins** notebook of the
**sqlite3, Deep Dive** guide wrote the same joins and groups in SQL, including the outer joins whose
unmatched rows this notebook adds first. The **Loading Strategies** notebook joined to load related
objects, a different purpose from joining to filter and count, and `contains_eager` is where the two
meet. Pandas, in the **Pandas, Deep Dive** guide, runs `groupby` on data that has already left the
database.

### What this notebook covers

- Two rows with nothing to join to
- Joins through relationships, and outer joins that keep the unmatched
- `group_by`, aggregates and `having`, with grade point averages in whole numbers
- `aliased`: one table twice in a query
- `any()` and `has()`: questions about related rows
- A subquery as a table
- `contains_eager`: a join that also loads a collection
- Which tool to use for which question
- Standing by program, finished
- Four errors, from a cartesian product to a collection filled with half its rows

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import ForeignKey, create_engine, func, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Course(Base):
    __tablename__ = "courses"
    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str]
    sections: Mapped[list["Section"]] = relationship()


class Section(Base):
    __tablename__ = "sections"
    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([Course(code="BIO-101", sections=[Section(), Section()]),
                     Course(code="STA-200", sections=[Section()]), Course(code="ART-100")])
    session.commit()
    inner = select(Course.code, func.count(Section.id)).join(Course.sections).group_by(Course.code)
    outer = (select(Course.code, func.count(Section.id)).outerjoin(Course.sections)
             .group_by(Course.code))
    print("join:     ", session.execute(inner.order_by(Course.code)).all())
    print("outerjoin:", session.execute(outer.order_by(Course.code)).all())
```

```
join:      [('BIO-101', 2), ('STA-200', 1)]
outerjoin: [('ART-100', 0), ('BIO-101', 2), ('STA-200', 1)]
```

One count of sections for every course, computed by the database. The inner join lost `ART-100`,
which has no sections, and the outer join kept it, with a count of 0: `COUNT` of a column counts the
rows where it is not `NULL`.


## Setup

Twelve imports, and the college built from its classes.

- `sqlalchemy` is the library itself, and the cell prints its version
- `func`, `case` and `distinct`, from `sqlalchemy`, write aggregates, conditions inside a query and
  `COUNT(DISTINCT ...)`, with `select`, `insert`, `create_engine` and `event`, and what the classes
  need
- `aliased` and `contains_eager`, from `sqlalchemy.orm`, name a table twice and load a collection
  from a join, and `selectinload` loads one in full, with `relationship` and the rest of the ORM
- `warnings` catches the warning that one of the Common errors produces, and `re` reads the table
  names out of it
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college from the classes of the **Relationships** notebook, where every student has
enrollments and every section has students. The first worked example adds one of each without.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import re
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, case, create_engine, distinct,
                        event, func, insert, select)
from sqlalchemy.orm import (DeclarativeBase, Mapped, Session, aliased, contains_eager, mapped_column, relationship,
                            selectinload, sessionmaker)
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### Two rows with nothing to join to

An outer join is only worth showing with rows that match nothing, and in Setup's college every
student has enrollments and every section has students. So the registrar admits Zoe Nakamura for the
autumn, with no courses yet, and opens a Fall 2026 section of Statistics that nobody has joined:


In [2]:
with SessionLocal.begin() as session:
    session.add(Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                        started_on=date(2026, 8, 24)))              # admitted for the autumn, with no enrollments yet
    statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
    fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
    session.add(fall)
    fall.sections.append(Section(course=statistics, capacity=30))     # open, and nobody enrolled yet

with SessionLocal() as session:
    print(session.scalar(select(func.count()).select_from(Student)), "students,",
          session.scalar(select(func.count()).select_from(Section)), "sections")


26 students, 41 sections


Twenty-six students, one of them with no enrollments, and forty-one sections, one of them empty.
Every outer join below has one of the two to keep.

### Joins through relationships, and outer joins that keep the unmatched

A count of enrollments for every student, joined through the relationship. The inner join and the
outer join differ by exactly the student who has none:


In [3]:
PER_STUDENT = select(Student.name, func.count(Enrollment.section_id)).group_by(Student.id).order_by(Student.id)

with SessionLocal() as session:
    inner = session.execute(PER_STUDENT.join(Student.enrollments)).all()
    outer = session.execute(PER_STUDENT.outerjoin(Student.enrollments)).all()
print("join:     ", len(inner), "students, the last", inner[-1])
print("outerjoin:", len(outer), "students, the last", outer[-1])


join:      25 students, the last ("Aoife O'Brien", 12)
outerjoin: 26 students, the last ('Zoe Nakamura', 0)


The inner join returned twenty-five students, and the outer join twenty-six, the last of them Zoe
Nakamura with 0. `func.count(Enrollment.section_id)` counts the rows where that column is not
`NULL`, which is why the outer join's empty match counted as 0 and not 1. `func.count()` alone
counts rows, and would have given Zoe Nakamura 1. The same holds for the empty section:


In [4]:
PER_SECTION = (
    select(Section.id, Term.name, Course.code, func.count(Enrollment.student_id).label("students"))
    .join(Section.term)
    .join(Section.course)
    .outerjoin(Section.enrollments)
    .group_by(Section.id)
    .where(Course.code == "STA-200")
    .order_by(Section.id)
)
with SessionLocal() as session:
    for row in session.execute(PER_SECTION):
        print(row)


(10, 'Fall 2024', 'STA-200', 3)
(20, 'Spring 2025', 'STA-200', 5)
(30, 'Fall 2025', 'STA-200', 8)
(40, 'Spring 2026', 'STA-200', 7)
(41, 'Fall 2026', 'STA-200', 0)


Every section of Statistics with its number of students, the Fall 2026 one among them with 0, kept
by the outer join to `enrollments`. The joins to `terms` and `courses` are inner joins, which cannot
lose a section, since every section has a term and a course.

### group_by, aggregates and having, with grade point averages in whole numbers

A grade point average is a sum of grade points times credits, over the credits. `POINTS` turns a
letter into its points with `case()`, in tenths, so that `A` is 40 and every sum is a whole number,
and the division into an average happens in Python, once for every student:


In [5]:
POINTS = case(                                     # grade points in tenths, so that every sum is a whole number
    {"A": 40, "A-": 37, "B+": 33, "B": 30, "B-": 27, "C+": 23, "C": 20, "C-": 17, "D": 10, "F": 0},
    value=Enrollment.grade,
)

GRADED = (
    select(Student.name,
           func.sum(POINTS * Course.credits).label("quality"),
           func.sum(Course.credits).label("credits"))
    .join(Student.enrollments)
    .join(Enrollment.section)
    .join(Section.course)
    .where(Enrollment.grade.is_not(None))
    .group_by(Student.id)
)
with SessionLocal() as session:
    for name, quality, credits in session.execute(GRADED.order_by(Student.id).limit(3)):
        print(f"{name:<13} {quality:>4} tenths over {credits} credits = {quality / credits / 10:.2f}")


Ana Reyes      776 tenths over 30 credits = 2.59
Ben Okafor     486 tenths over 20 credits = 2.43
Chloe Martin   191 tenths over 10 credits = 1.91


Ana Reyes has 776 tenths of a point over 30 credits, an average of 2.59, the same figure the
**Reading Results** notebook worked out in Python. `.label()` names a column for the query to use
later. `having()` keeps groups the way `where()` keeps rows, and whole numbers make an exact test: an
average of at least 3.0 is `quality >= 30 * credits`:


In [6]:
AT_LEAST_THREE = GRADED.having(func.sum(POINTS * Course.credits) >= 30 * func.sum(Course.credits))
with SessionLocal() as session:
    print([name for name, quality, credits in session.execute(AT_LEAST_THREE.order_by(Student.name))])


['Felix Wagner']


One student's whole record averages 3.0 or more, Felix Wagner's, exactly 3.0 over the one term Felix
has finished, which the **Declarative Models** notebook's dean's list showed at 3.00. The `HAVING`
clause repeats the sum rather than the label, since a `HAVING` that names a label can find a table's
column of the same name first, as the **Everyday Requests** notebook of the **sqlite3, Deep Dive**
guide showed.

### aliased: one table twice in a query

Chloe Martin's classmates in Spring 2026 are the students with an enrollment in one of Chloe's
sections. That needs `enrollments` twice, Chloe's and the classmates', and `aliased` gives each
one a name of its own:


In [7]:
chloes = aliased(Enrollment)
theirs = aliased(Enrollment)
CLASSMATES = (
    select(Student.name, func.count().label("sections_shared"))
    .join(theirs, theirs.student_id == Student.id)
    .join(chloes, chloes.section_id == theirs.section_id)
    .where(chloes.student_id == 3, theirs.student_id != 3, chloes.section_id > 30)
    .group_by(Student.id)
    .order_by(func.count().desc(), Student.name)
)
print(" ".join(str(CLASSMATES.compile(engine)).split()))
with SessionLocal() as session:
    print(session.execute(CLASSMATES.limit(4)).all())


SELECT students.name, count(*) AS sections_shared FROM students JOIN enrollments AS enrollments_1 ON enrollments_1.student_id = students.id JOIN enrollments AS enrollments_2 ON enrollments_2.section_id = enrollments_1.section_id WHERE enrollments_2.student_id = ? AND enrollments_1.student_id != ? AND enrollments_2.section_id > ? GROUP BY students.id ORDER BY count(*) DESC, students.name
[('Maya Patel', 3), ('Wes Carter', 3), ('Felix Wagner', 2), ('Jonas Berg', 2)]


The SQL shows `enrollments` twice under two aliases, `enrollments_1` and `enrollments_2`, joined on
the section they share. Maya Patel and Wes Carter share all three of Chloe Martin's Spring 2026
sections, and others share two. The joins here name their conditions, since both sides are the same
table and a relationship could not say which enrollment is Chloe's.

### any() and has(): questions about related rows

`any()` asks whether some related row matches a condition, and `has()` whether the one related object
does. Neither adds rows to the result, since both are `EXISTS` subqueries:


In [8]:
EVER_FAILED = select(Student.name).where(Student.enrollments.any(Enrollment.grade == "F")).order_by(Student.name)
NO_ENROLLMENTS = select(Student.name).where(~Student.enrollments.any())
HISTORY_ENROLLMENTS = (select(func.count()).select_from(Enrollment)
                       .where(Enrollment.student.has(Student.program == "History")))

print(" ".join(str(EVER_FAILED.compile(engine)).split()))
with SessionLocal() as session:
    failed = session.scalars(EVER_FAILED).all()
    print(len(failed), "students have failed a course, first", failed[:3])
    print("students with no enrollments:", session.scalars(NO_ENROLLMENTS).all())
    print("enrollments of History students:", session.scalar(HISTORY_ENROLLMENTS))


SELECT students.name FROM students WHERE EXISTS (SELECT 1 FROM enrollments WHERE students.id = enrollments.student_id AND enrollments.grade = ?) ORDER BY students.name
13 students have failed a course, first ["Aoife O'Brien", 'Chloe Martin', 'Daniel Kim']
students with no enrollments: ['Zoe Nakamura']
enrollments of History students: 48


Thirteen students have an F somewhere, and each appears once however many they have, since `EXISTS`
answers yes or no. `~Student.enrollments.any()`, with no condition, finds the students with no
enrollment at all: Zoe Nakamura. `has()` is the same test from the many to one side, for the
enrollments whose student is in History.

### A subquery as a table

A query can be the `FROM` of another. `.subquery()` turns one into something to join to, with its
columns in `.c`, which is how a report counts per program something it first computed per student:


In [9]:
per_student = GRADED.with_only_columns(
    Student.id.label("student_id"),
    func.sum(POINTS * Course.credits).label("quality"),
    func.sum(Course.credits).label("credits"),
).subquery()

BY_PROGRAM = (
    select(Student.program, func.count(), func.sum(per_student.c.quality), func.sum(per_student.c.credits))
    .join(per_student, per_student.c.student_id == Student.id)
    .group_by(Student.program)
    .order_by(Student.program)
)
with SessionLocal() as session:
    for program, students, quality, credits in session.execute(BY_PROGRAM):
        print(f"{program:<17} {students} students, averaging {quality / credits / 10:.2f}")


Biology           5 students, averaging 2.67
Computer Science  5 students, averaging 2.51
History           5 students, averaging 1.97
Mathematics       5 students, averaging 2.30
Psychology        5 students, averaging 2.27


The inner query summed every student's points and credits, and the outer query grouped those rows by
program, so every program's average weighs every credit equally. Zoe Nakamura has no graded courses,
so the subquery has no row for that student, and the join leaves Computer Science's count at five.

### contains_eager: a join that also loads a collection

A query that joins to enrollments to filter them already has the enrollments' rows.
`contains_eager()` tells SQLAlchemy to fill each student's `enrollments` from those rows instead of
loading them again:


In [10]:
SPRING_WITH_ENROLLMENTS = (
    select(Student)
    .join(Student.enrollments)
    .join(Enrollment.section)
    .where(Section.term_id == 4, Student.program == "History")
    .options(contains_eager(Student.enrollments))
    .order_by(Student.name, Enrollment.section_id)
    .execution_options(populate_existing=True)
)
with SessionLocal() as session:
    for student in session.scalars(SPRING_WITH_ENROLLMENTS).unique():
        print(f"{student.name:<14}", [enrollment.section_id for enrollment in student.enrollments])


Aoife O'Brien  [32, 35, 39]
Elena Petrova  [32, 35, 39]
Jonas Berg     [34, 37, 40]
Olivia Brandt  [32, 35, 39]
Tara Nilsen    [34, 37, 40]


One query, and every History student's collection filled from its rows: but only with Spring 2026's
enrollments, because those are the rows the `WHERE` kept. That is what `contains_eager` is for, and
the last of the Common errors shows the same collection used as though it were complete.
`populate_existing` makes the query overwrite collections a session may already hold, so a filtered
one is never mixed with a full one.

### Which tool to use for which question

| Use | When | Why |
|---|---|---|
| `.join(Cls.relationship)` | rows that must match on both sides | the condition comes from the foreign key |
| `.outerjoin(Cls.relationship)` | rows that may match nothing, which must stay | unmatched rows kept with `NULL` |
| `func.count(column)` after an outer join | counting matches, with 0 for none | `COUNT` of a column skips `NULL` |
| `func.count(distinct(column))` | counting one side of a join that multiplies rows | every value counted once |
| `having()` | keeping or dropping whole groups | `where()` runs before the grouping, `having()` after |
| `aliased(Cls)` | the same table twice in one query | a second name for the second use |
| `any()` and `has()` | a yes or no about related rows | an `EXISTS` subquery that adds no rows |
| `.subquery()` | a result to join to, such as a total for every student | a query used as a table |
| `contains_eager()` | a collection that the query's own join has already filtered | no second query, and a collection that holds only what the join kept |

The defaults are joins through relationships, outer joins wherever an unmatched row must be counted,
and `any()` rather than a join whenever the question is only whether something exists.

### Standing by program, finished

The pieces of this notebook in one function. `standing` computes every student's grade points and
credits for one term in a subquery, grouped by student, and then, grouped by program, counts the
students, the ones whose term average is at least 3.0 and the ones below 2.0, with `case()` inside a
`SUM` and whole numbers throughout:


In [11]:
def standing(session, term):
    """For every program: how many students took graded courses in a term, and how many of them made the
    dean's list, a term average of 3.0 or more, or fell below 2.0."""
    per_student = (
        select(Enrollment.student_id,
               func.sum(POINTS * Course.credits).label("quality"),
               func.sum(Course.credits).label("credits"))
        .join(Enrollment.section)
        .join(Section.course)
        .join(Section.term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
        .group_by(Enrollment.student_id)
        .subquery()
    )
    deans_list = case((per_student.c.quality >= 30 * per_student.c.credits, 1), else_=0)
    below_two = case((per_student.c.quality < 20 * per_student.c.credits, 1), else_=0)
    report = (
        select(Student.program, func.count(), func.sum(deans_list), func.sum(below_two))
        .join(per_student, per_student.c.student_id == Student.id)
        .group_by(Student.program)
        .order_by(Student.program)
    )
    return session.execute(report).all()



for term in ("Spring 2025", "Fall 2025"):
    print(term)
    with SessionLocal() as session:
        for program, students, deans, below in standing(session, term):
            print(f"    {program:<17} {students} students, {deans} on the dean's list, {below} below 2.0")


Spring 2025
    Biology           3 students, 2 on the dean's list, 0 below 2.0
    Computer Science  4 students, 0 on the dean's list, 0 below 2.0
    History           4 students, 0 on the dean's list, 2 below 2.0
    Mathematics       3 students, 0 on the dean's list, 0 below 2.0
    Psychology        3 students, 0 on the dean's list, 1 below 2.0
Fall 2025
    Biology           5 students, 2 on the dean's list, 0 below 2.0
    Computer Science  5 students, 2 on the dean's list, 0 below 2.0
    History           5 students, 0 on the dean's list, 3 below 2.0
    Mathematics       5 students, 0 on the dean's list, 3 below 2.0
    Psychology        5 students, 0 on the dean's list, 3 below 2.0


One statement for every term, and the database did all the counting. Fall 2025 has every student who
had started by then, all twenty-five, and Spring 2025 only those who had started before it,
seventeen. `case()` inside a `SUM` counts the rows that meet a condition, one of the most useful
shapes in reporting. The counts agree with the **Declarative Models** notebook's dean's list for the
same terms.

### Where each part came from

| In `standing` | What it relies on | The section that showed it |
|---|---|---|
| `.join(Enrollment.section).join(Section.course).join(Section.term)` | joins through relationships | Joins through relationships, and outer joins that keep the unmatched |
| `func.sum(POINTS * Course.credits)` and `group_by` | aggregates, in whole numbers | group_by, aggregates and having, with grade point averages in whole numbers |
| `.subquery()` joined on `student_id` | a query used as a table | A subquery as a table |
| `func.sum(case(...))` | counting the rows that meet a condition | Standing by program, finished |
| `quality >= 30 * credits` | an exact test of an average | group_by, aggregates and having, with grade point averages in whole numbers |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/14-joins-and-aggregates-solutions.ipynb).

**1.** Count the students in every Spring 2026 section with an outer join, and list the sections with
fewer than eight.


In [12]:
# your code here


**2.** With `any()`, list the students who have an A in some course, and count them.


In [13]:
# your code here


**3.** With `has()`, count the enrollments in Mathematics department courses. The path is an
enrollment's section, and the section's course.


In [14]:
# your code here


**4.** Compute every course's number of grades given and its average grade in tenths of a point, with
`group_by` and `having` keeping the courses with at least fifteen grades.


In [15]:
# your code here


**5.** With `aliased`, find the pairs of students in the same program who started on the same day,
counting each pair once.


In [16]:
# your code here


**6.** Run `standing` for Fall 2024, and add up the students across the programs.


In [17]:
# your code here


## Common errors

### sqlalchemy.exc.SAWarning: SELECT statement has a cartesian product between FROM element(s) "students" and FROM element "courses".  Apply join condition(s) between each element to resolve.


In [18]:
HISTORY_COURSES_TAKEN = select(Student.name, Course.code).where(Course.department == "History")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with SessionLocal() as session:
        rows = session.execute(HISTORY_COURSES_TAKEN).all()
print(len(rows), "rows, the first", rows[:2])
for warning in caught:
    tables = sorted(re.findall(r'"(\w+)"', str(warning.message)))
    print(type(warning.message).__name__ + ": a cartesian product between the tables", " and ".join(tables))


26 rows, the first [('Ana Reyes', 'HIS-110'), ('Ben Okafor', 'HIS-110')]


The query was meant to list who took a History course, and it names `students` and `courses` with
nothing to join them, so every student was paired with World History, whether or not they ever took
it: twenty-six rows, one for every student, Zoe Nakamura included, who has taken nothing. SQLAlchemy
noticed while compiling, and warned. The cell caught the warning, since one printed the ordinary way
carries the name of a temporary file, and printed its two table names sorted, because SQLAlchemy
takes them from a set and can name them in either order from one run to the next. Join through the
relationships that connect the two, and ask for every name once, since a student can take World
History in more than one term:


In [19]:
HISTORY_COURSES_TAKEN = (
    select(Student.name)
    .join(Student.enrollments)
    .join(Enrollment.section)
    .join(Section.course)
    .where(Course.department == "History")
    .distinct()
    .order_by(Student.name)
)
with SessionLocal() as session:
    names = session.scalars(HISTORY_COURSES_TAKEN).all()
print(len(names), "students have taken World History, the first", names[:2])


22 students have taken World History, the first ['Ana Reyes', "Aoife O'Brien"]


### No error, and counts multiplied: two collections joined at once


In [20]:
SECTIONS_AND_ENROLLMENTS = (
    select(Course.code, func.count(Section.id).label("sections"), func.count(Enrollment.student_id).label("enrollments"))
    .join(Course.sections)
    .outerjoin(Section.enrollments)
    .group_by(Course.code)
    .order_by(Course.code)
)
with SessionLocal() as session:
    print(session.execute(SECTIONS_AND_ENROLLMENTS.limit(3)).all())


[('BIO-101', 23, 23), ('CHE-110', 23, 23), ('CSC-101', 23, 23)]


Introduction to Biology has four sections, and the query says twenty-three: joining the sections to
their enrollments made one row for every enrollment, and `COUNT(sections.id)` counted a section once
for every student in it. Nothing raised, and the number is plausible, which is what makes it
dangerous. Count the multiplied side with `DISTINCT`:


In [21]:
SECTIONS_AND_ENROLLMENTS = (
    select(Course.code, func.count(distinct(Section.id)).label("sections"),
           func.count(Enrollment.student_id).label("enrollments"))
    .join(Course.sections)
    .outerjoin(Section.enrollments)
    .group_by(Course.code)
    .order_by(Course.code)
)
with SessionLocal() as session:
    print(session.execute(SECTIONS_AND_ENROLLMENTS.limit(3)).all())


[('BIO-101', 4, 23), ('CHE-110', 4, 23), ('CSC-101', 4, 23)]


### AttributeError: Neither 'InstrumentedAttribute' object nor 'Comparator' object associated with Student.enrollments has an attribute 'grade'


In [22]:
select(Student.name).where(Student.enrollments.grade == "F")


AttributeError: Neither 'InstrumentedAttribute' object nor 'Comparator' object associated with Student.enrollments has an attribute 'grade'

`Student.enrollments` is a collection, and a collection has no single `grade` to compare: the
question "a student whose enrollments' grade is F" means "a student with some enrollment whose grade
is F", which is `any()`. The message names both objects that were asked for a `grade` and had none:


In [23]:
with SessionLocal() as session:
    print(session.scalar(select(func.count()).select_from(Student).where(Student.enrollments.any(Enrollment.grade == "F"))))


13


### No error, and half a transcript: a collection filled by contains_eager, used as complete


In [24]:
FAILED_WITH_ENROLLMENTS = (
    select(Student)
    .join(Student.enrollments)
    .where(Enrollment.grade == "F")
    .options(contains_eager(Student.enrollments))
    .order_by(Student.name)
    .execution_options(populate_existing=True)
)
with SessionLocal() as session:
    for student in session.scalars(FAILED_WITH_ENROLLMENTS).unique().all()[:3]:
        print(f"{student.name:<14} enrollments: {len(student.enrollments)}")


Aoife O'Brien  enrollments: 2
Chloe Martin   enrollments: 1
Daniel Kim     enrollments: 2


The query found the students with an F and filled every student's `enrollments` from the joined
rows, which were only the failed ones: Aoife O'Brien has twelve enrollments, and this session says
two, the two with an F. Any code that later reads `student.enrollments` in this session, to count
credits or build a transcript, gets the filtered list and no warning. Use `contains_eager` only
where the filtered collection is what the code wants, and otherwise filter with `any()` and load the
collection in full:


In [25]:
FAILED = (
    select(Student)
    .where(Student.enrollments.any(Enrollment.grade == "F"))
    .options(selectinload(Student.enrollments))
    .order_by(Student.name)
    .execution_options(populate_existing=True)
)
with SessionLocal() as session:
    for student in session.scalars(FAILED).all()[:3]:
        print(f"{student.name:<14} enrollments: {len(student.enrollments)}")


Aoife O'Brien  enrollments: 12
Chloe Martin   enrollments: 6
Daniel Kim     enrollments: 12


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [26]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A join through a relationship writes its own condition; an outer join keeps the rows with nothing
  to match, and `COUNT` of a column counts them as 0.
- Aggregates with `group_by` give one row for every group, `having()` keeps or drops groups, and
  grade points in tenths keep every sum a whole number that every SQLite adds up the same.
- `aliased` names a table a second time, and a subquery becomes a table to join to.
- `any()` and `has()` ask about related rows without multiplying the result, and a join across two
  collections multiplies it unless the count is `DISTINCT`.
- Two tables without a join make a cartesian product, which SQLAlchemy warns about, and
  `contains_eager` fills a collection with only the rows the join kept.


## What is next

The **Cascades and Deletes** notebook deletes objects that others refer to: `delete-orphan` against
`passive_deletes`, and the `SELECT` that loads every child just to delete it.


---

&#8592; **Previous:** [Loading Strategies](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/13-loading-strategies.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
